# Tarea 4. PSO y abejas
## MA2015 - Diseño de Algoritmos Bioinspirados

**Autoras:** Viviana Carrizales Luna (A01286191) & Steffany Mishell Lara Muy (A00838589)

*Instituto Tecnológico y de Estudios Superiores de Monterrey (ITESM)*

---

## Functions

In [44]:
import math
def f1(args):
    x = args[0]
    try: return abs(x * math.sin(x) / (2 * x - 5))
    except ZeroDivisionError: return float('-inf')

def f2(args):
    x = args[0]
    return math.exp((x ** 3) - x)

def f3(args):
    x, y = args[0], args[1]
    return -3 * y / (x**2 + y**2 + 1)

def f4(args):
    x, y = args[0], args[1]
    term1 = 3 * (1 - x)**2 * math.exp(-(x**2) - (y + 1)**2)
    term2 = 10 * (x / 5 - x**3 - y**5) * math.exp(-x**2 - y**2)
    term3 = (1 / 3) * math.exp(-(x + 1)**2 - y**2)
    return term1 - term2 - term3


In [ ]:
import random
import time
import math

# Parámetros PSO
PSO_N = 30           
PSO_W = 0.5          
PSO_C1 = 1.5         
PSO_C2 = 1.5         
PSO_VMAX = 2.0       
PSO_V_INIT_TXT = "Uniforme(-1, 1)" 

ITERACIONES = 100
ABC_LIMIT = 50       





def initialize_pso(n_particles, dimensions, bounds):
    swarm = []
    for _ in range(n_particles):
        position = []
        velocity = []
        for d in range(dimensions):
            min_val, max_val = bounds[d]
            position.append(random.uniform(min_val, max_val))
            # Velocidad inicial entre -1 y 1
            velocity.append(random.uniform(-1, 1))
            
        particle = {
            'position': position,
            'velocity': velocity,
            'best_position': position[:],
            'best_value': float('-inf') # Maximización
        }
        swarm.append(particle)
    return swarm

def evaluate(particle, objective_function):
    value = objective_function(particle['position'])
    if value > particle['best_value']: # Maximización
        particle['best_value'] = value
        particle['best_position'] = particle['position'][:]
    return value

def update_velocity(particle, global_best_position, w, c1, c2, v_max):
    new_velocity = []
    for i in range(len(particle['position'])):
        r1 = random.random()
        r2 = random.random()
        
        current_vel = particle['velocity'][i]
        p_best = particle['best_position'][i]
        curr_pos = particle['position'][i]
        g_best = global_best_position[i]
        
        inertia = w * current_vel
        cognitive = c1 * r1 * (p_best - curr_pos)
        social = c2 * r2 * (g_best - curr_pos)
        
        vel = inertia + cognitive + social
        
        # Aplicar V_MAX
        if vel > v_max: vel = v_max
        elif vel < -v_max: vel = -v_max
            
        new_velocity.append(vel)
    particle['velocity'] = new_velocity

def update_position(particle, bounds):
    new_position = []
    for i in range(len(particle['position'])):
        min_bound, max_bound = bounds[i]
        new_pos = particle['position'][i] + particle['velocity'][i]
        
        # Rebote
        if new_pos < min_bound:
            new_pos = min_bound
            particle['velocity'][i] *= -1 
        elif new_pos > max_bound:
            new_pos = max_bound
            particle['velocity'][i] *= -1
            
        new_position.append(new_pos)
    particle['position'] = new_position

def pso_main(objective_function, bounds, n_particles, n_iterations, w, c1, c2, v_max):
    dimensions = len(bounds)
    t0 = time.time()
    
    swarm = initialize_pso(n_particles, dimensions, bounds)
    global_best_position = None
    global_best_value = float('-inf')
    
    for _ in range(n_iterations):
        for particle in swarm:
            value = evaluate(particle, objective_function)
            if value > global_best_value:
                global_best_value = value
                global_best_position = particle['position'][:]
        
        for particle in swarm:
            update_velocity(particle, global_best_position, w, c1, c2, v_max)
            update_position(particle, bounds)
            
    t1 = time.time()
    return global_best_position, global_best_value, (t1 - t0)


def run_abc(func, bounds):
    dimensions = len(bounds)
    n_sources = PSO_N // 2 
    colony = []
    for _ in range(n_sources):
        pos = [random.uniform(b[0], b[1]) for b in bounds]
        colony.append({'pos': pos, 'val': func(pos), 'trials': 0})
    
    gbest_val = max(s['val'] for s in colony)
    gbest_pos = next(s['pos'] for s in colony if s['val'] == gbest_val)[:]
    
    for _ in range(ITERACIONES):
        for i in range(n_sources): _abc_search(i, colony, bounds, func)
        
        vals = [s['val'] for s in colony]
        min_v = min(vals); shift = abs(min_v) + 0.1 if min_v < 0 else 0
        total = sum(v + shift for v in vals); probs = [(v + shift)/total if total !=0 else 1.0/len(vals) for v in vals]
        
        for _ in range(n_sources):
            r = random.random(); cum = 0; sel = 0
            for idx, p in enumerate(probs):
                cum += p
                if r <= cum: sel = idx; break
            _abc_search(sel, colony, bounds, func)
            
        for i in range(n_sources):
            if colony[i]['trials'] > ABC_LIMIT:
                colony[i]['pos'] = [random.uniform(b[0], b[1]) for b in bounds]
                colony[i]['val'] = func(colony[i]['pos']); colony[i]['trials'] = 0
                
        curr = max(colony, key=lambda x: x['val'])
        if curr['val'] > gbest_val:
            gbest_val = curr['val']; gbest_pos = curr['pos'][:]
            
    return gbest_pos, gbest_val, 0

def _abc_search(idx, colony, bounds, func):
    k = idx
    while k == idx: k = random.randint(0, len(colony)-1)
    phi = random.uniform(-1, 1); j = random.randint(0, len(bounds)-1)
    new_pos = colony[idx]['pos'][:]; new_pos[j] += phi * (new_pos[j] - colony[k]['pos'][j])
    mn, mx = bounds[j]; new_pos[j] = max(mn, min(mx, new_pos[j]))
    new_val = func(new_pos)
    if new_val > colony[idx]['val']:
        colony[idx]['pos'] = new_pos; colony[idx]['val'] = new_val; colony[idx]['trials'] = 0
    else: colony[idx]['trials'] += 1



[-] Configuración General:
    Población: 30 partículas/abejas
    Iteraciones: 100
------------------------------------------------------------

>>> EXPERIMENTO 1: f1 (1D)
  [PSO] Parámetros usados:
        w=0.5, c1=1.5, c2=1.5
        Vel. Inicial: Uniforme(-1, 1) | Vel. Máxima: 2.0
  [PSO] RESULTADOS:
        > Mejor Posición: [2.5000]
        > Mejor Valor:    105284333014774.156250
        > Tiempo Ejec.:   0.00367 seg
----------------------------------------
  [ABC] RESULTADOS:
        > Mejor Posición: [2.5000]
        > Mejor Valor:    5297324931560.225586
        > Tiempo Ejec.:   0.00639 seg
  [CONCLUSIÓN f1 (1D)]:
        Ganador en Calidad: PSO

>>> EXPERIMENTO 2: f2 (1D)
  [PSO] Parámetros usados:
        w=0.5, c1=1.5, c2=1.5
        Vel. Inicial: Uniforme(-1, 1) | Vel. Máxima: 2.0
  [PSO] RESULTADOS:
        > Mejor Posición: [-0.5774]
        > Mejor Valor:    1.469468
        > Tiempo Ejec.:   0.00374 seg
----------------------------------------
  [ABC] RESULTADOS:
  

In [ ]:
functions = [f1, f2, f3, f4]
function_names = ["f1 (1D)", "f2 (1D)", "f3 (2D)", "f4 (Peaks)"]

bounds_list = [
    [(0, 14)],            
    [(-1, 0)],              
    [(-5, 5), (-5, 5)],     
    [(-3, 3), (-3, 3)]     
]



print(f"[-] Configuración General:")
print(f"    Población: {PSO_N} partículas/abejas")
print(f"    Iteraciones: {ITERACIONES}")
print("-" * 60)

for i, func in enumerate(functions):
    print(f"\n>>> EXPERIMENTO {i+1}: {function_names[i]}")
    
    pso_pos, pso_val, pso_time = pso_main(
        func, bounds_list[i], PSO_N, ITERACIONES, PSO_W, PSO_C1, PSO_C2, PSO_VMAX
    )

    print(f"  [PSO] Parámetros usados:")
    print(f"        w={PSO_W}, c1={PSO_C1}, c2={PSO_C2}")
    print(f"        Vel. Inicial: {PSO_V_INIT_TXT} | Vel. Máxima: {PSO_VMAX}")
    print(f"  [PSO] RESULTADOS:")

    pos_str = ", ".join([f"{x:.4f}" for x in pso_pos])
    print(f"        > Mejor Posición: [{pos_str}]")
    print(f"        > Mejor Valor:    {pso_val:.6f}")
    print(f"        > Tiempo Ejec.:   {pso_time:.5f} seg")
    
    print("-" * 40)
    

    t0 = time.time()
    abc_pos, abc_val, _ = run_abc(func, bounds_list[i])
    abc_time = time.time() - t0
    

    print(f"  [ABC] RESULTADOS:")
    pos_str_abc = ", ".join([f"{x:.4f}" for x in abc_pos])
    print(f"        > Mejor Posición: [{pos_str_abc}]")
    print(f"        > Mejor Valor:    {abc_val:.6f}")
    print(f"        > Tiempo Ejec.:   {abc_time:.5f} seg")

    print(f"  [CONCLUSIÓN {function_names[i]}]:")
    winner = "PSO" if pso_val >= abc_val else "ABC"
    print(f"        Ganador en Calidad: {winner}")
    print("="*60)
